In [1]:
import warnings
warnings.filterwarnings('ignore')

import os
import pandas as pd
import numpy as np
from tqdm import tqdm
import pickle

# fast parquet
try:
    import fastparquet
except:
    ! pip install fastparquet
    import fastparquet

# binning
try:
    from optbinning import OptimalBinning
except:
    ! pip install optbinning
    from optbinning import OptimalBinning

#### Functions

In [2]:
def bin_features(list_cols, X, y):
    dict_bins = {}
    list_cols_scorecard = []
    for col in tqdm(list_cols):        
        # init
        cls_binning = OptimalBinning(
            name=col,
            dtype='numerical',
            solver='cp',
            monotonic_trend=None,
            prebinning_method='cart',
            user_splits=None,
            user_splits_fixed=None,
        )
        # fit
        cls_binning.fit(
            X[col],
            y,
        )
        # assign
        dict_bins[col] = cls_binning
    # return
    return dict_bins

#### Constants

In [3]:
str_project = os.getcwd().split('/')[4].replace('_','-')
print(f'Project: {str_project}')

str_task = os.getcwd().split('/')[5]
print(f'Task: {str_task}')

str_dirname_output = './output'

str_target = 'has_inst_tag'

list_str_inst = [
    # from ben: 2025-02-21
    'CURRENT',
    'SELF',
    # key words
    'CHIME-STRIDE',
    'CHIMEFINAL',
    # from dustin: 2025-02-24
    'SELF FIN',
    'SELF/LEAD',
    'SELFINC/LEAD',
    'SBNASELFLNDR',
    'SBNA SELF',
    'CHIME',
    'CLEO',
    'CLEO AI',
    'VARO',
    'ATLAS',
    'ATLCAPBKSELF',
    'POSSIBLE',
    'POSSIBLE FIN',
    'KIKOFF',
    'SUPER.COM',
    'STEP',
    'STEP MOBILE',
    'BRIGHT',
    'BRIGHT BLDR',
    'FIG TECH INC',
    'SELF/RENT',
    'SELFBILLSE',
    'PROGRESSRES',
    'FLEX',
    'FLEXFINANCE',
]

# rm no variance
list_cols_novariance = [
    'sr04s__tu',
    'sr60s__tu',
    'sd04s__tu',
    'sd05s__tu',
    'sd64s__tu',
    'sd71s__tu',
    'sd72s__tu',
    'sd85s__tu',
    'sd86s__tu',
    'sd87s__tu',
    'sd50s__tu',
    'linkc012__tu',
    'linkc010__tu',
    'linkc051__tu',
    'fltinsuredlifepremium__app',
    'fltinsuredunemploymentamount__app',
    'fltinsuredunemploymentpremium__app',
    'fltaddfee__app',
    'fltinsureddisabilitypremium__app',
    'sp04s__tu',
    'sp06s__tu',
    'sp07s__tu',
    'sp60s__tu',
    'sg05s__tu',
    'sp71s__tu',
    'hr29s__tu',
    'lienjudgmentforeclosurecount__ln',
    'purchaseactivityindex__ln',
    'inputprovidedfirstname__ln',
    'inputprovidedlastname__ln',
    'inputprovideddateofbirth__ln',
    'addrcurrentcorrectional__ln',
    'addrpreviouscorrectional__ln',
    's208s__tu'
]

# force features
list_cols_force = [
    'ENG-franchise',
    'ENG-loan_to_value',
    'ENG-bk',
    'ENG-wtd_avg',
    'fltgrossmonthly__income_sum',
    'miles_odometer__app',
    'ENG-vehicle_age',
    'g232s__tu',
    'rp01s__tu',
    'g106s__tu',
    'au20s__tu'
    
]
# odometer
# ltv
# bk
# franchise
# wtd avg
# veh age

# rm cols with all nans
list_cols_nan = [
    'linkb006__tu',
    'linkb007__tu',
     'linkc014__tu',
     'linkc015__tu',
     'linkc016__tu',
     'linkc022__tu',
     'linkc023__tu',
     'approvaldate__app',
     'fundeddate__app',
     'dtmapproved__app',
     'dtmdeclined__app',
     'defaultdate__app',
     'chargeoffdate__app',
     'defaultamount__app',
     'chargeoffamount__app',
     'bittrade__app',
     'purchaseactivitycount__ln',
     'purchaseactivitydollartotal__ln',
     'attribute_index__ln',
     'bkc203__tu',
     'bkc204__tu',
     'bkc205__tu',
     'bkc222__tu',
     'bkc223__tu',
     'bkc224__tu',
     'bkc225__tu',
     'bkc202__tu',
     'bkc201__tu',
     'bkc231__tu',
     'bkc232__tu',
     'bkc234__tu',
     'bkc235__tu',
     'bkc252__tu',
     'bkc253__tu',
     'bkc254__tu',
     'bkc255__tu',
     'bkc233__tu',
]

# cols with IDs to remove
list_ids = [
    'bigdealertypeid__app',
    'bigstatusid__app',
    'bigdealerid__app',
    'biglnriskviewattributesv5id__ln',
    'biglnriskviewscoreid__ln',
    'inputprovidedlexid__ln',    
]

# dict monotone constraints
dict_monotone_constraints = None

int_n_feats = 200

Project: 20250221-credit-builder-analysis
Task: 05_feat_analysis


#### Output directory

In [4]:
try:
    os.mkdir(str_dirname_output)
except:
    pass

#### Load in data

In [5]:
str_filename = 'df.gzip'
str_uri = f's3://20241112-simple-model-test/08_prep_data/{str_filename}'
df = pd.read_parquet(
    str_uri,
)
# sort
df.sort_values(by='request_datetime', ascending=True, inplace=True)
df

,accountid,request_datetime,response_model_name,file_key,bitdebtor,bitdebtor__app,dealerstate__app,strdealershiptrackertype__app,strname__app,bitdealertrack__app,...,ENG-franchise,ENG-has_codebtor,ENG-vehicle_age,ENG-payment_to_income,ENG-loan_to_value,ENG-bk,ENG-perfect_payment_hx,ENG-perfect_payment_hx_open,ENG-perfect_payment_hx_closed,ENG-bk_x_wtd_avg
0,5702434,2021-07-26 16:29:29.3903686,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Iowa,Franchise,Iowa,True,...,1,0,7,NaN,1.311220,1,0,0,0,0.616667
1,5714239,2021-07-26 16:39:34.1121025,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Utah,Franchise,Utah,True,...,1,0,5,NaN,1.432368,0,0,0,0,NaN
2,5713063,2021-07-26 16:48:39.3211104,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Illinois,Franchise,Illinois,True,...,1,0,4,NaN,1.587073,1,0,0,0,NaN
3,5713732,2021-07-27 09:02:35.3300974,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Michigan,Independent,Michigan,False,...,0,0,2,NaN,1.081881,0,1,0,1,0.000000
4,5715634,2021-07-27 09:18:12.2190097,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Arizona,Franchise,Arizona,False,...,1,0,4,NaN,1.371350,0,1,0,1,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94472,8420588,2024-11-26 06:16:16+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1,North Carolina,Independent,North Carolina,True,...,0,0,4,NaN,1.280957,0,0,0,0,0.000000
94473,8401043,2024-11-26 06:21:44+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1,Nevada,Franchise,Nevada,True,...,1,0,0,NaN,1.198869,1,0,0,0,0.367073
94474,8414683,2024-11-26 06:25:09+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1,Virginia,Franchise,Virginia,False,...,1,0,3,NaN,1.313231,0,0,0,0,NaN
94475,8359085,2024-11-26 06:32:28+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1,Alabama,Franchise,Alabama,True,...,1,0,3,NaN,0.991586,1,0,0,0,0.006585


#### Create target

In [6]:
df['list_institutions'] = df['str_institution__tu_pmthx'].apply(
    lambda x: eval(x.replace('nan','None')),
)
df['list_institutions'] = df['list_institutions'].apply(
    lambda x: [] if x is None else x,
)

list_str_col_new = []
for str_inst in tqdm(list_str_inst):
    str_col_new = f'{str_inst}_tag'
    df[str_col_new] = df['list_institutions'].apply(
        lambda x: 1 if str_inst in x else 0,
    )
    list_str_col_new.append(str_col_new)

df['sum'] = df[list_str_col_new].sum(axis=1)
df['has_inst_tag'] = df['sum'].apply(
    lambda x: 1 if x > 0 else 0,
)
flt_mn = df['has_inst_tag'].mean()
print(f'Proportion has tag: {flt_mn:0.4f}')
# show
df

100%|██████████| 29/29 [00:01<00:00, 20.24it/s]


Proportion has tag: 0.2053


,accountid,request_datetime,response_model_name,file_key,bitdebtor,bitdebtor__app,dealerstate__app,strdealershiptrackertype__app,strname__app,bitdealertrack__app,...,BRIGHT_tag,BRIGHT BLDR_tag,FIG TECH INC_tag,SELF/RENT_tag,SELFBILLSE_tag,PROGRESSRES_tag,FLEX_tag,FLEXFINANCE_tag,sum,has_inst_tag
0,5702434,2021-07-26 16:29:29.3903686,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Iowa,Franchise,Iowa,True,...,0,0,0,0,0,0,0,0,0,0
1,5714239,2021-07-26 16:39:34.1121025,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Utah,Franchise,Utah,True,...,0,0,0,0,0,0,0,0,0,0
2,5713063,2021-07-26 16:48:39.3211104,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Illinois,Franchise,Illinois,True,...,0,0,0,0,0,0,0,0,0,0
3,5713732,2021-07-27 09:02:35.3300974,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Michigan,Independent,Michigan,False,...,0,0,0,0,0,0,0,0,1,1
4,5715634,2021-07-27 09:18:12.2190097,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Arizona,Franchise,Arizona,False,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94472,8420588,2024-11-26 06:16:16+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1,North Carolina,Independent,North Carolina,True,...,0,0,0,0,0,0,0,0,1,1
94473,8401043,2024-11-26 06:21:44+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1,Nevada,Franchise,Nevada,True,...,0,0,0,0,0,0,0,0,0,0
94474,8414683,2024-11-26 06:25:09+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1,Virginia,Franchise,Virginia,False,...,0,0,0,0,0,0,0,0,0,0
94475,8359085,2024-11-26 06:32:28+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1,Alabama,Franchise,Alabama,True,...,0,0,0,0,0,0,0,0,1,1


#### Load in df feature importance

In [7]:
# save
str_filename = 'df_feat_imp.csv'
str_local_path = f'../04_credit_builder_feats/{str_dirname_output}/{str_filename}'
df_tmp = pd.read_csv(str_local_path)
df_tmp.columns = ['feature', 'No', 'Yes', 'description']

# make rank
df_tmp['rank'] = range(0, df_tmp.shape[0])

# dict
dict_rank = dict(zip(df_tmp['feature'], df_tmp['rank']))

# list_cols 
list_cols_use = df_tmp['feature'].tolist()

# # rm non-numeric
list_cols_use = [col for col in list_cols_use if df[col].dtype in ['int64','float64']]

# # rm id cols
list_cols_use = [i for i in list_cols_use if i not in list_ids]

# remove cols with no variance
list_cols_use = [i for i in list_cols_use if i not in list_cols_novariance]

# remove cols with all nans
list_cols_use = [i for i in list_cols_use if i not in list_cols_nan]

# use top n
list_cols_use = list_cols_use[:int_n_feats]

# add forced features and remove dupes
list_cols_use = list(set(list_cols_use + list_cols_force))

# subset df
df_tmp = df[list_cols_use]

# show 
df_tmp

,addrinputphonecount__ln,bi20s__tu,hi35s__tu,ENG-wtd_avg,bc97b__tu,s043s__tu,linkt004__tu,ENG-loan_to_value,se20s__tu,jt40s__tu,...,of57s__tu,scg044c__tu,sl04s__tu,sl00s__tu,agg404__tu,agg402__tu,g106s__tu,in34s__tu,in27s__tu,bi36s__tu
0,1.0,NaN,NaN,0.616667,NaN,1.0,1.0,1.311220,32.0,2.0,...,0.0,NaN,NaN,NaN,437.0,200.0,282.0,118.0,2.0,NaN
1,0.0,NaN,NaN,NaN,NaN,NaN,2.0,1.432368,112.0,NaN,...,0.0,NaN,NaN,NaN,NaN,0.0,282.0,84.0,NaN,NaN
2,0.0,NaN,NaN,NaN,NaN,NaN,0.0,1.587073,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,426.0,NaN,NaN,NaN
3,0.0,NaN,NaN,1.000000,NaN,2.0,0.0,1.081881,45.0,0.0,...,NaN,NaN,NaN,NaN,721.0,450.0,439.0,121.0,2.0,NaN
4,0.0,NaN,NaN,1.000000,NaN,NaN,1.0,1.371350,NaN,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,343.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94472,0.0,34.0,NaN,0.030636,0.0,NaN,0.0,1.280957,64.0,NaN,...,NaN,0.0,0.0,23.0,0.0,0.0,268.0,102.0,1.0,999.0
94473,0.0,NaN,NaN,0.367073,0.0,NaN,3.0,1.198869,NaN,NaN,...,NaN,NaN,NaN,9.0,NaN,NaN,417.0,NaN,NaN,NaN
94474,0.0,NaN,NaN,NaN,NaN,NaN,0.0,1.313231,NaN,NaN,...,NaN,NaN,NaN,5.0,0.0,0.0,395.0,NaN,NaN,NaN
94475,0.0,34.0,NaN,0.006585,NaN,1.0,1.0,0.991586,34.0,NaN,...,NaN,0.0,0.0,23.0,25.0,75.0,206.0,101.0,NaN,12.0


In [8]:
[col for col in list(df_tmp.columns) if 'ENG' in col]

['ENG-wtd_avg',
 'ENG-loan_to_value',
 'ENG-franchise',
 'ENG-vehicle_age',
 'ENG-bk']

#### Get bins

In [9]:
dict_bins = bin_features(
    list_cols=list_cols_use,
    X=df_tmp,
    y=df['has_inst_tag'],
)
# pickle dict_bins
str_filename = 'dict_bins.pkl'
str_local_path = f'./output/{str_filename}'
pickle.dump(dict_bins, open(str_local_path, 'wb'))

100%|██████████| 210/210 [00:04<00:00, 42.35it/s]


#### Show bins in order of importance

In [10]:
# only get the top n features from feature importance because we don't have room

# show binning table
list_df = []
for key, val in tqdm(dict_bins.items()):
    if key in list_cols_use:
        df_bins = val.binning_table.build()
        list_df.append(df_bins)
        df_bins['Feature'] = key
    else:
        pass

# concat
df = pd.concat(list_df)

# get the aa dict
list_cols = [
    'feature_name',
    'Description',
]
str_filename = 'data_dictionary.csv'
str_local_path = f'./{str_filename}'
df_dd = pd.read_csv(str_local_path, usecols=list_cols)

# rename
dict_rename = {
    'feature_name': 'Feature',
    'Description': 'Description',
}
df_dd.rename(columns=dict_rename, inplace=True)

# join
df = pd.merge(
    left=df,
    right=df_dd,
    on='Feature',
    how='left',
)

# reorder
list_cols = [
    'Feature',
    'Bin',
    'Count',
    'Count (%)',
    'Non-event',
    'Event',
    'Event rate',
    'WoE',
    'IV',
    'JS',
    'Description',
]
df = df[list_cols].copy()

# rank
df['rank'] = df['Feature'].map(dict_rank)

# sort
df.sort_values(by='rank', ascending=True, inplace=True)

# save
str_filename = f'df_bins_top_{int_n_feats}.csv'
str_local_path = f'{str_dirname_output}/{str_filename}'
df.to_csv(str_local_path, index=False)

# show
df

100%|██████████| 210/210 [00:00<00:00, 345.00it/s]


,Feature,Bin,Count,Count (%),Non-event,Event,Event rate,WoE,IV,JS,Description,rank
1122,g201a__tu,"[3418.50, inf)",4726,0.050023,2375,2351,0.497461,-1.343409,0.120349,1.400552e-02,Total open to buy of open trades verified in p...,0
1124,g201a__tu,Missing,44498,0.470993,41680,2818,0.063329,1.340428,0.549349,6.394942e-02,Total open to buy of open trades verified in p...,0
1121,g201a__tu,"[1305.50, 3418.50)",8612,0.091154,3732,4880,0.566651,-1.621766,0.327444,3.696269e-02,Total open to buy of open trades verified in p...,0
1125,g201a__tu,,94477,1.000000,75082,19395,0.205288,,1.152454,1.335422e-01,Total open to buy of open trades verified in p...,0
1120,g201a__tu,"[500.50, 1305.50)",8038,0.085079,4534,3504,0.435929,-1.095866,0.131808,1.569814e-02,Total open to buy of open trades verified in p...,0
...,...,...,...,...,...,...,...,...,...,...,...,...
668,fltgrossmonthly__income_sum,"[7644.15, inf)",6559,0.069424,5369,1190,0.181430,0.153123,0.001555,1.941321e-04,NaN,2621
667,fltgrossmonthly__income_sum,"[6668.07, 7644.15)",5032,0.053262,3984,1048,0.208267,-0.018163,0.000018,2.208019e-06,NaN,2621
665,fltgrossmonthly__income_sum,"[5422.89, 6075.01)",7437,0.078718,5916,1521,0.204518,0.004727,0.000002,2.195553e-07,NaN,2621
664,fltgrossmonthly__income_sum,"[5000.68, 5422.89)",6549,0.069318,5148,1401,0.213926,-0.052144,0.000191,2.391863e-05,NaN,2621
